In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(
    "/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env"
)

# api_key = os.getenv("CO_API_KEY")
# print(api_key)

load_dotenv(dotenv_path=os.path.join("..", ".env"))

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY not found. Add it to the project-root .env file."
)

# The client reads OPENAI_API_KEY from the environment automatically.
client = OpenAI()

MODEL = "gpt-6-luna"  # strong general/coding model; or "gpt-6-luna" for fast, high-volume tasks
print(f"Client ready. Using model: {MODEL}")

Client ready. Using model: gpt-6-luna


# OpenAI Prompt Engineering Patterns (Responses API)

This notebook demonstrates common prompt types and patterns using the OpenAI Python SDK (`client.responses.create` and `client.responses.stream`).

## 1. Zero-Shot Prompting
Asking the model to perform a task directly without giving any prior examples.

In [4]:
response = client.responses.create(
    model=MODEL,
    max_output_tokens=200,
    input=(
        "Classify the sentiment of this review as Positive, Neutral, or Negative:\n\n"
        "'The packaging was damaged, though the product inside works is also damaged. I am really disappointed with the quality and service. I expected better.'"
    ),
)

print(response.output_text)

Negative.


## 2. System / Instruction Prompting (Role & Constraints)
Using `instructions` to define persona, tone, and formatting constraints.

In [5]:
response = client.responses.create(
    model=MODEL,
    max_output_tokens=250,
    instructions=(
        "You are a  maths researcher. Please answer the following question in brief and provide examples."
    ),
    input="How does the Pythagorean theorem work? Can you provide a proof and an example of its application?",
)

print(response.output_text)

The **Pythagorean theorem** applies to a right-angled triangle. If its two shorter sides (the legs) have lengths \(a\) and \(b\), and its longest side (the hypotenuse) has length \(c\), then
\[
a^2+b^2=c^2.
\]
In other words, the sum of the squares of the legs equals the square of the hypotenuse.

**Proof:** Draw an altitude from the right-angle vertex to the hypotenuse, splitting it into lengths \(x\


## 3. Few-Shot Prompting
Providing input-output examples inside the prompt so the model follows an exact structure and pattern.

In [6]:
few_shot_prompt = """Solve each math problem and return the operation type and final answer.

Problem: What is 15% of 240?
Output: {"operation": "percentage", "answer": 36}

Problem: Solve for x: 3x + 7 = 22
Output: {"operation": "linear_equation", "answer": 5}

Problem: Find the area of a circle with a radius of 4 (round to 2 decimal places).
Output:"""

response = client.responses.create(
    model=MODEL,
    max_output_tokens=150,
    input=few_shot_prompt,
)

print(response.output_text)

{"operation": "area_circle", "answer": 50.27}


## 4. Chain-of-Thought (CoT) Prompting
Guiding the model to break down reasoning step-by-step before giving the final answer.

In [7]:
cot_prompt = """A clinical trial enrolls 240 patients across 3 sites.
Site A enrolls twice as many patients as Site B.
Site C enrolls 40 fewer patients than Site A.
How many patients are enrolled at each site?

Think through the steps carefully before stating the final numbers."""

response = client.responses.create(
    model=MODEL,
    max_output_tokens=400,
    input=cot_prompt,
)

print(response.output_text)

Let Site B enroll \(x\) patients. Then Site A enrolls \(2x\), and Site C enrolls \(2x-40\).

\[
x+2x+(2x-40)=240
\]
\[
5x=280 \quad\Rightarrow\quad x=56
\]

So Site B enrolls **56**, Site A **112**, and Site C **72** patients. Together, they total 240.


## 5. Structured Output Prompting (JSON Schema)
Enforcing a strict JSON schema with `text={'format': ...}` so responses parse cleanly into Python dicts.

In [8]:
import json

response = client.responses.create(
    model=MODEL,
    input="Summarize the key details of the movie Inception (2010).",
    text={
        "format": {
            "type": "json_schema",
            "name": "movie_summary",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "title": {"type": "string"},
                    "release_year": {"type": "integer"},
                    "director": {"type": "string"},
                    "premise": {"type": "string"},
                    "main_themes": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
                "required": [
                    "title",
                    "release_year",
                    "director",
                    "premise",
                    "main_themes",
                ],
                "additionalProperties": False,
            },
        }
    },
)

parsed = json.loads(response.output_text)
print(json.dumps(parsed, indent=2))

{
  "title": "Inception",
  "release_year": 2010,
  "director": "Christopher Nolan",
  "premise": "Dom Cobb is a skilled thief who steals secrets from people\u2019s dreams. Offered a chance to return to his family, he accepts a difficult assignment: plant an idea in a target\u2019s mind by guiding a team through several layers of dreams.",
  "main_themes": [
    "Dreams and reality",
    "Memory and grief",
    "Guilt and forgiveness",
    "The power of ideas",
    "Family and belonging"
  ]
}


## 7. Multi-Turn Conversational Prompting
Maintaining context across turns by passing conversation history (or chaining via `previous_response_id`).

In [9]:
conversation = [
    {
        "role": "user",
        "content": "What are the main differences between LLM and Agents?",
    },
]

res1 = client.responses.create(
    model=MODEL,
    max_output_tokens=250,
    input=conversation,
)
print("Turn 1:\n", res1.output_text)

# Append assistant reply and ask a follow-up question
conversation.append({"role": "assistant", "content": res1.output_text})
conversation.append(
    {
        "role": "user",
        "content": "When would it make sense to combine both approaches in the same application?",
    },
)

res2 = client.responses.create(
    model=MODEL,
    max_output_tokens=150,
    input=conversation,
)
print("\nTurn 2:\n", res2.output_text)

Turn 1:
 An **LLM** is a language model that generates or interprets text. An **agent** is a system that uses a model—often an LLM—to pursue a goal by choosing actions and responding to their results.

| | LLM | Agent |
|---|---|---|
| **What it is** | A model that predicts and produces text | A system that uses a model to carry out a task |
| **Typical behavior** | Responds to a prompt | Plans, acts, observes results, and may try again |
| **Tools** | Usually only generates text, unless connected to tools | May use tools such as web search, code execution, or APIs |
| **State** | Often handles one prompt at a time | May keep task context or memory across steps |
| **Example** | “Here’s how to find a flight

Turn 2:
 Combine them when an application needs both **natural-language understanding or generation** and **actions that take multiple steps**.

For example, in a customer-support app:

- The **LLM


## 8. Prompt Chaining (Multi-Step Pipeline)
Feeding the output of one prompt into the next prompt to build reliable multi-stage workflows.

In [ ]:
raw_notes = """
Weekend notes: Want to finish two Python practice exercises on dictionaries,
review 20 flashcards on REST APIs, and spend 90 minutes on Sunday afternoon
recording a short demo video. Saturday morning is free, and Sunday morning has a 1-hour errand.
"""

# Step 1: Extract the list of tasks and available time slots
step1 = client.responses.create(
    model=MODEL,
    max_output_tokens=200,
    input=f"List the tasks and time constraints from these notes:\n{raw_notes}",
)
summary = step1.output_text
print("--- Step 1: Extracted Tasks & Constraints ---")
print(summary)

# Step 2: Turn the extracted summary into a simple two-day schedule
step2 = client.responses.create(
    model=MODEL,
    max_output_tokens=500,
    instructions="Organize the input into a clean Saturday/Sunday schedule in bullet points.",
    input=summary,
)
print("\n--- Step 2: Weekend Schedule ---")
print(step2.output_text)

--- Step 1: Extracted Tasks & Constraints ---
**Tasks**
- Finish two Python practice exercises on dictionaries.
- Review 20 flashcards on REST APIs.
- Record a short demo video for 90 minutes on Sunday afternoon.

**Time constraints**
- Saturday morning is free.
- Sunday morning includes a 1-hour errand.

--- Step 2: Weekend Schedule ---
- **Saturday morning**
  - 9:00–10:00 — Finish Python dictionary practice exercise 1



## 9. Streaming Prompt Responses
Streaming text chunks in real time using `client.responses.stream`.

In [ ]:
with client.responses.stream(
    model=MODEL,
    max_output_tokens=256,
    input="Explain what a 483 observation is in three concise sentences.",
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)